In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
def load_all_fixtures(data_folder="data"):
    """
    Load ALL fixtures from all seasons & stack into one DataFrame.
    """
    all_fixtures = []

    for season in os.listdir(data_folder):
        season_path = os.path.join(data_folder, season)
        if not os.path.isdir(season_path):
            continue

        fx_path = os.path.join(season_path, "fixtures.csv")
        if not os.path.exists(fx_path):
            print(f"⚠ fixtures.csv missing for season {season}")
            continue

        df_fx = pd.read_csv(fx_path)
        df_fx["season"] = season
        all_fixtures.append(df_fx)

    if not all_fixtures:
        raise ValueError("❌ No fixtures found in data/ folder.")

    fixtures = pd.concat(all_fixtures, ignore_index=True)
    print(f"✔ Loaded {len(fixtures):,} fixture rows across seasons")
    return fixtures


In [3]:
def load_all_teams():
    """
    Load team IDs + names for each season from master_team_list.csv.
    Normalizes columns automatically.
    """
    teams = pd.read_csv("master_team_list.csv")
    teams.columns = [c.strip().lower() for c in teams.columns]

    # Expected columns: season, team, team_name
    teams = teams.rename(columns={"team": "id", "team_name": "name"})
    print(f"✔ Loaded {len(teams):,} team mappings")
    return teams[["season", "id", "name"]]


In [4]:
def build_fixture_lookup(fixtures, teams):
    """
    Merge fixtures with team names so we know:
    team_h_name, team_a_name
    """
    fx = fixtures.merge(
        teams.rename(columns={"id": "team_h", "name": "team_h_name"}),
        on=["season", "team_h"],
        how="left"
    ).merge(
        teams.rename(columns={"id": "team_a", "name": "team_a_name"}),
        on=["season", "team_a"],
        how="left"
    )

    fx = fx[[
        "season", "event", "team_h", "team_a",
        "team_h_name", "team_a_name",
        "team_h_difficulty", "team_a_difficulty"
    ]]

    print(f"✔ Fixture lookup ready ({len(fx):,} rows)")
    return fx


In [5]:
def fix_opponent_vectorized(df, fx):
    """
    Uses vectorized pandas merge logic to reconstruct opponent fields.
    No loops. ~10x faster.
    """
    print("🔄 Matching HOME games...")
    df_home = df.merge(
        fx,
        left_on=["season", "Gameweek", "Player Team Name"],
        right_on=["season", "event", "team_h_name"],
        how="left",
        suffixes=("", "_fxh")
    )

    print("🔄 Matching AWAY games...")
    df_full = df_home.merge(
        fx,
        left_on=["season", "Gameweek", "Player Team Name"],
        right_on=["season", "event", "team_a_name"],
        how="left",
        suffixes=("", "_fxa")
    )

    print("🔧 Constructing final opponent data...")

    # Determine home/away
    df_full["Is Home"] = df_full["team_h"].notna()

    # Opponent ID
    df_full["Opponent ID"] = df_full["team_a"].where(df_full["Is Home"], df_full["team_h_fxa"])

    # Opponent Name
    df_full["Opponent Name"] = df_full["team_a_name"].where(df_full["Is Home"], df_full["team_h_name_fxa"])

    # Opponent Difficulty
    df_full["Opponent Difficulty"] = df_full["team_h_difficulty"].where(
        df_full["Is Home"],
        df_full["team_a_difficulty_fxa"]
    )

    print("✔ Opponent reconstruction complete")
    return df_full


In [6]:
def generate_diagnostics(df_fixed, path="output/opponent_fix_diagnostics.csv"):
    """
    Identify rows where opponent data is missing.
    """
    issues = df_fixed[df_fixed["Opponent ID"].isna()]

    if len(issues) > 0:
        issues.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"⚠ {len(issues):,} rows missing opponent data saved to:")
        print(f"   {path}")
    else:
        print("✔ No missing opponent data detected")

    return issues


In [7]:
print("📥 Loading training data...")
df = pd.read_csv("output/training_data.csv")
print(f"✔ Loaded {len(df):,} training rows")

print("\n📥 Loading fixtures...")
fixtures = load_all_fixtures()

print("\n📥 Loading teams...")
teams = load_all_teams()

print("\n🔧 Building fixture lookup...")
fx = build_fixture_lookup(fixtures, teams)

print("\n⚡ Running vectorized opponent reconstruction...")
df_fixed = fix_opponent_vectorized(df, fx)

print("\n📊 Running diagnostics...")
issues = generate_diagnostics(df_fixed)

print("\n💾 Saving fixed dataset...")
df_fixed.to_csv("output/training_data_fixed.csv", index=False, encoding="utf-8-sig")
print("✔ Saved to output/training_data_fixed.csv")


📥 Loading training data...
✔ Loaded 168,972 training rows

📥 Loading fixtures...
✔ Loaded 3,040 fixture rows across seasons

📥 Loading teams...
✔ Loaded 160 team mappings

🔧 Building fixture lookup...
✔ Fixture lookup ready (3,040 rows)

⚡ Running vectorized opponent reconstruction...
🔄 Matching HOME games...
🔄 Matching AWAY games...
🔧 Constructing final opponent data...
✔ Opponent reconstruction complete

📊 Running diagnostics...
⚠ 32,515 rows missing opponent data saved to:
   output/opponent_fix_diagnostics.csv

💾 Saving fixed dataset...
✔ Saved to output/training_data_fixed.csv


In [8]:
df_fixed[
    (df_fixed["Player Name"] == "Sokratis Papastathopoulos") &
    (df_fixed["season"] == "2019-20")
]


,A,Player UUID,Code,Player Name,Web Name,Player Team Name,season,Gameweek,Minutes Played,Goals Scored,...,team_a_name,team_h_difficulty,team_a_difficulty,event_fxa,team_h_fxa,team_a_fxa,team_h_name_fxa,team_a_name_fxa,team_h_difficulty_fxa,team_a_difficulty_fxa
1039,150366,05b195d0-ea16-4d04-bc9b-8567821f271b,5,Sokratis Papastathopoulos,Sokratis,Arsenal,2019-20,1,90,0,...,NaN,NaN,NaN,1.0,13.0,1.0,Newcastle,Arsenal,3.0,2.0
1069,150367,05b195d0-ea16-4d04-bc9b-8567821f271b,5,Sokratis Papastathopoulos,Sokratis,Arsenal,2019-20,2,90,0,...,Burnley,3.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1099,150368,05b195d0-ea16-4d04-bc9b-8567821f271b,5,Sokratis Papastathopoulos,Sokratis,Arsenal,2019-20,3,90,0,...,NaN,NaN,NaN,3.0,10.0,1.0,Liverpool,Arsenal,3.0,5.0
1129,150369,05b195d0-ea16-4d04-bc9b-8567821f271b,5,Sokratis Papastathopoulos,Sokratis,Arsenal,2019-20,4,90,0,...,Spurs,3.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1159,150370,05b195d0-ea16-4d04-bc9b-8567821f271b,5,Sokratis Papastathopoulos,Sokratis,Arsenal,2019-20,5,90,0,...,NaN,NaN,NaN,5.0,18.0,1.0,Watford,Arsenal,3.0,3.0
1190,150371,05b195d0-ea16-4d04-bc9b-8567821f271b,5,Sokratis Papastathopoulos,Sokratis,Arsenal,2019-20,6,90,0,...,Aston Villa,2.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1221,150372,05b195d0-ea16-4d04-bc9b-8567821f271b,5,Sokratis Papastathopoulos,Sokratis,Arsenal,2019-20,7,90,0,...,NaN,NaN,NaN,7.0,12.0,1.0,Man Utd,Arsenal,3.0,4.0
1252,150373,05b195d0-ea16-4d04-bc9b-8567821f271b,5,Sokratis Papastathopoulos,Sokratis,Arsenal,2019-20,8,90,0,...,Bournemouth,2.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1283,150374,05b195d0-ea16-4d04-bc9b-8567821f271b,5,Sokratis Papastathopoulos,Sokratis,Arsenal,2019-20,9,90,0,...,NaN,NaN,NaN,9.0,15.0,1.0,Sheffield Utd,Arsenal,3.0,3.0
1314,150375,05b195d0-ea16-4d04-bc9b-8567821f271b,5,Sokratis Papastathopoulos,Sokratis,Arsenal,2019-20,10,90,1,...,Crystal Palace,3.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
